# 02 — Feature Search and Graph Experiments

Продолжение исследования: восстановление промежуточного состояния, автоматический поиск и отбор признаков, RFE и графовые признаки.

> Для полного повторного запуска нужны промежуточные файлы предыдущего этапа. Итоги выполненных запусков сохранены отдельно в `results/`.

---

## Исходное описание этапа

# Продолжение работы с сохранённой версии (Kaggle)

Этот блокнот **восстанавливает состояние** по файлам, сохранённым в Output предыдущей версии, и продолжает работу:
граф-признаки и сравнение с пулом v2, агенты (по желанию), итоговые таблицы.

## Как запустить
1. Откройте блокнот в Kaggle (File → Import Notebook, если загружаете файл `.ipynb`).
2. Правая панель → **Add Input** → вкладка с вашими блокнотами (**Notebook Output Files**) → выберите блокнот с последней сохранённой версией. Его файлы появятся в `/kaggle/input/...`.
3. Нужные файлы: `base_listens.parquet`, `features_snap_u1500_s42.parquet`, `features_wide_u1500_s42.parquet`, `features_v2_u1500_s42.parquet`, `feature_sets_v1.json`, `selected_sets_v1.json` (поиск идёт по всему `/kaggle/input` и `/kaggle/working` автоматически).
4. Для графа нужен исходный `multi_event.parquet` (если его нет среди входных файлов, он скачается заново, ~384 МБ, нужен Internet: On).
5. Запускайте **Save Version → Save & Run All (Commit)**, чтобы сессия не прерывалась. Сначала можно прогнать с `DEBUG = True`.

Каждая тяжёлая стадия сохраняет результат на диск (`results/`, `features_graph_v1.parquet`), поэтому при повторном запуске уже готовое не пересчитывается.

In [ ]:
import os, glob, json, time, re, gc, warnings
import numpy as np, pandas as pd, pyarrow.parquet as pq, lightgbm as lgb
warnings.filterwarnings("ignore", message="Overriding the parameters")

DEBUG = False            # True: облегчённый пробный прогон
RUN_AGENTS = False       # True: заново запустить агентов (random/evo/surrogate), результаты кэшируются в results/
SEED, N_USERS, POS_THRESHOLD = 42, 1500, 50
DAY, SHIFT, TRAIN_DAYS = 86400, 26, 60
SEARCH_ROOTS = ["/kaggle/input", "/kaggle/working", "."]

def find_file(name, required=True):
    for root in SEARCH_ROOTS:
        hits = glob.glob(os.path.join(root, "**", name), recursive=True)
        if hits:
            return hits[0]
    if required:
        raise FileNotFoundError(f"не найден файл {name}: подключите Output предыдущей версии через Add Input")
    return None

if DEBUG:
    ROUNDS, PER_ROUND, SEEDS_A = 2, 3, [0]
else:
    ROUNDS, PER_ROUND, SEEDS_A = 10, 8, list(range(10))
os.makedirs("results", exist_ok=True)

## 1. Загрузка данных и кэшей признаков

In [ ]:
BASE = find_file("base_listens.parquet")
df = pd.read_parquet(BASE)
print("df:", df.shape, "| колонки:", list(df.columns))
df["pr"] = (np.minimum(df["played_ratio_pct"], 100) / 100).astype(np.float32)
df["skip"] = (df["pr"] < 0.1).astype(np.float32)
df["full"] = (df["pr"] >= 0.9).astype(np.float32)

t_val, t_test = df["timestamp"].quantile([0.8, 0.9])
chk = np.where(df["timestamp"] < t_val, "train", np.where(df["timestamp"] < t_test, "val", "test"))
assert (chk == df["split"].to_numpy()).all(), "разбиение не совпадает с сохранённым"

FS  = pd.read_parquet(find_file(f"features_snap_u{N_USERS}_s{SEED}.parquet"))
FW  = pd.read_parquet(find_file(f"features_wide_u{N_USERS}_s{SEED}.parquet"))
FV2 = pd.read_parquet(find_file(f"features_v2_u{N_USERS}_s{SEED}.parquet"))
assert len(FS) == len(FW) == len(FV2) == len(df), "кэши не совпадают с df по числу строк"
FEATS = list(FS.columns)
POOL2 = pd.concat([FS[[c for c in FEATS if c.startswith(("item_", "gitem_"))]], FW, FV2], axis=1)
POOL2_COLS = list(POOL2.columns)

sets_json = json.load(open(find_file("feature_sets_v1.json")))
M2_COLS = list(sets_json["M2"])
assert sorted(sets_json["M1"]) == sorted(FEATS) and POOL2_COLS == sets_json["POOL2"], "наборы признаков не совпадают с кэшами"
sel_json = json.load(open(find_file("selected_sets_v1.json")))
sets_rfe = {int(k): v for k, v in sel_json["RFE"].items()}
sets_a1  = {int(k): v for k, v in sel_json["A1"].items()}
print(len(FEATS), len(M2_COLS), len(POOL2_COLS), "| RFE K:", sorted(sets_rfe))

# исполнитель и альбом
def load_map(fname):
    p = find_file(fname, required=False)
    if p is None:
        from huggingface_hub import hf_hub_download
        p = hf_hub_download(repo_id="yandex/yambda", repo_type="dataset", filename=fname, local_dir="data")
    m = pq.read_table(p).to_pandas()
    other = [c for c in m.columns if c != "item_id"]
    assert len(other) == 1
    return m.drop_duplicates("item_id").set_index("item_id")[other[0]]
for col, fname_ in [("artist_id", "artist_item_mapping.parquet"), ("album_id", "album_item_mapping.parquet")]:
    s = df["item_id"].map(load_map(fname_)).astype("float64")
    df[col] = np.where(s.isna(), -1 - df["item_id"].astype(np.int64), s).astype(np.int64)
print("артистов/альбомов в df:", df["artist_id"].nunique(), df["album_id"].nunique())

## 2. Функции (метрика, обучение, отбор, генерация признаков)

In [ ]:
def add_window_feats2(df, keys, windows_days, name, target="label", snap=DAY, with_cnt=True, recency=False):
    codes = df.groupby(keys, sort=False).ngroup().to_numpy(np.int64)
    ts = df["timestamp"].to_numpy(np.int64)
    val = df[target].to_numpy(np.float64)
    tq = ts if snap is None else (ts // snap) * snap
    comb = (codes << SHIFT) | ts
    order = np.argsort(comb, kind="stable")
    comb_s, codes_s, tq_s, val_s = comb[order], codes[order], tq[order], val[order]
    cum = np.concatenate([[0.0], np.cumsum(val_s)])
    hi = np.searchsorted(comb_s, (codes_s << SHIFT) | tq_s, side="left")
    tag = "rate" if target == "label" else f"mean_{target}"
    out = {}
    for w in windows_days:
        lo = np.searchsorted(comb_s, (codes_s << SHIFT) | np.maximum(tq_s - w * DAY, 0), side="left")
        cnt = (hi - lo).astype(np.float64)
        mean = np.where(cnt > 0, (cum[hi] - cum[lo]) / np.maximum(cnt, 1), np.nan)
        if with_cnt:
            r = np.empty(len(df), np.float32); r[order] = cnt; out[f"{name}_cnt_{w}d"] = r
        r = np.empty(len(df), np.float32); r[order] = mean; out[f"{name}_{tag}_{w}d"] = r
    if recency:
        prev = hi - 1
        ok = prev >= 0
        prev = np.where(ok, prev, 0)
        ok &= codes_s[prev] == codes_s
        last_ts = comb_s[prev] & ((1 << SHIFT) - 1)
        rec = np.where(ok, (tq_s - last_ts) / DAY, np.nan)
        r = np.empty(len(df), np.float32); r[order] = rec; out[f"{name}_recency_d"] = r
    return pd.DataFrame(out, index=df.index)

def ndcg_at_k(groups, scores, labels, k=10, min_size=10):
    d = pd.DataFrame({"g": np.asarray(groups), "s": np.asarray(scores), "y": np.asarray(labels, dtype=float)})
    d = d[d.groupby("g")["y"].transform("size") >= min_size]
    a = d.sort_values(["g", "s"], ascending=[True, False]); ra = a.groupby("g").cumcount().to_numpy() + 1
    dcg = pd.Series(a["y"].to_numpy() / np.log2(ra + 1) * (ra <= k), index=a.index).groupby(a["g"]).sum()
    b = d.sort_values(["g", "y"], ascending=[True, False]); rb = b.groupby("g").cumcount().to_numpy() + 1
    idcg = pd.Series(b["y"].to_numpy() / np.log2(rb + 1) * (rb <= k), index=b.index).groupby(b["g"]).sum()
    return (dcg / idcg).dropna().mean()

def ndcg_groups(groups, scores, labels, k=10, min_size=10):
    d = pd.DataFrame({"g": np.asarray(groups), "s": np.asarray(scores), "y": np.asarray(labels, dtype=float)})
    d = d[d.groupby("g")["y"].transform("size") >= min_size]
    a = d.sort_values(["g", "s"], ascending=[True, False]); ra = a.groupby("g").cumcount().to_numpy() + 1
    dcg = pd.Series(a["y"].to_numpy() / np.log2(ra + 1) * (ra <= k), index=a.index).groupby(a["g"]).sum()
    b = d.sort_values(["g", "y"], ascending=[True, False]); rb = b.groupby("g").cumcount().to_numpy() + 1
    idcg = pd.Series(b["y"].to_numpy() / np.log2(rb + 1) * (rb <= k), index=b.index).groupby(b["g"]).sum()
    return (dcg / idcg).dropna()

def prep(mask):
    d = df.loc[mask, ["uid", "timestamp", "label"]].copy()
    d["g"] = d["uid"].astype(np.int64) * 1000 + d["timestamp"].astype(np.int64) // DAY
    d = d.sort_values("g", kind="stable")
    return d, d.groupby("g", sort=True).size().to_numpy()

params_r = dict(objective="lambdarank", metric="ndcg", eval_at=[10], learning_rate=0.05, num_leaves=63,
                min_data_in_leaf=200, feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, verbose=-1)

def fit_sel(cols, seed=0, rounds=300, max_bin=63):
    ci = [CP[c] for c in cols]
    dtr  = lgb.Dataset(Xtr_p[:, ci], d_tr["label"].to_numpy(), group=sz_tr, params={"max_bin": max_bin})
    dsel = lgb.Dataset(Xsel_p[:, ci], d_sel["label"].to_numpy(), group=sz_sel, reference=dtr)
    m = lgb.train({**params_r, "seed": seed}, dtr, rounds, valid_sets=[dsel],
                  callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(0)])
    p_sel = m.predict(Xsel_p[:, ci], num_iteration=m.best_iteration)
    p_rep = m.predict(Xrep_p[:, ci], num_iteration=m.best_iteration)
    s_sel = ndcg_at_k(d_sel["g"], p_sel, d_sel["label"])
    s_rep = ndcg_at_k(d_rep["g"], p_rep, d_rep["label"])
    del dtr, dsel; gc.collect()
    return m, s_sel, s_rep, p_rep

def boot_diff_rep(pred_a, pred_b, n_boot=1000, seed=0):
    ga = ndcg_groups(d_rep["g"], pred_a, d_rep["label"]); gb = ndcg_groups(d_rep["g"], pred_b, d_rep["label"])
    diff = (gb - ga).dropna()
    pu = diff.groupby(diff.index // 1000).agg(["sum", "count"])
    sums, cnts = pu["sum"].to_numpy(), pu["count"].to_numpy()
    r = np.random.default_rng(seed); est = []
    for _ in range(n_boot):
        idx = r.integers(0, len(sums), len(sums)); est.append(sums[idx].sum() / cnts[idx].sum())
    return diff.mean(), np.percentile(est, [2.5, 97.5])

def rfe_path(start_cols, targets, drop_frac=0.15):
    cols = list(start_cols); out = {}
    for K in sorted(targets, reverse=True):
        while len(cols) > K:
            m = fit_sel(cols)[0]
            imp = pd.Series(m.feature_importance("gain"), index=cols).sort_values(ascending=False)
            cols = imp.index[:max(K, int(len(cols) * (1 - drop_frac)))].tolist()
        out[K] = list(cols)
    return out

## 3. Восстановление выборок и базовых моделей

Разбиение val на sel/rep (seed 123), матрицы признаков и базовые модели M1, M2, пул v2 (3 обучения) — как в исходных экспериментах.

In [ ]:
tr = (df["split"] == "train") & (df["timestamp"] >= t_val - TRAIN_DAYS * DAY)
va = df["split"] == "val"
d_tr, sz_tr = prep(tr)
rs = np.random.default_rng(123)
val_users = df.loc[va, "uid"].unique()
half = set(rs.choice(val_users, size=len(val_users) // 2, replace=False).tolist())
is_sel = df["uid"].isin(half).to_numpy()
d_sel, sz_sel = prep(va & is_sel)
d_rep, sz_rep = prep(va & ~is_sel)
print("групп sel/rep:", len(sz_sel), len(sz_rep), "| пересечение пользователей:", len(set(d_sel.uid) & set(d_rep.uid)))

idx_tr_all, idx_sel, idx_rep = d_tr.index.to_numpy(), d_sel.index.to_numpy(), d_rep.index.to_numpy()
XP = POOL2.to_numpy(dtype=np.float32)
CP = {c: i for i, c in enumerate(POOL2_COLS)}
Xtr_p, Xsel_p, Xrep_p = XP[idx_tr_all], XP[idx_sel], XP[idx_rep]
rnd = ndcg_at_k(d_rep["g"], np.random.default_rng(0).random(len(d_rep)), d_rep["label"])
v = XP[idx_rep, CP["gitem_rate_30d"]]
pop = ndcg_at_k(d_rep["g"], np.where(np.isnan(v), 0.64, v), d_rep["label"])
print(f"rep: случайный {rnd:.4f} | популярность {pop:.4f}")
del XP, POOL2, FS, FW, FV2; gc.collect()          # освобождаем память: всё нужное уже в Xtr_p/Xsel_p/Xrep_p

R = {}
for name, cols in [("M1", FEATS), ("M2", M2_COLS), ("POOL2", POOL2_COLS)]:
    m, s_sel, s_rep, p = fit_sel(list(cols))
    R[name] = dict(m=m, sel=s_sel, rep=s_rep, p=p, n=len(cols))
    print(f"{name:6s} ({len(cols)}): NDCG@10 sel={s_sel:.4f} rep={s_rep:.4f}", flush=True)

## 4. Агенты (язык признаков, компилятор, три поставщика предложений)

Агент не пишет код: он возвращает JSON-описание признака, который считает наша проверенная функция, поэтому утечки из будущего невозможны.
Результаты каждого запуска кэшируются в `results/agent_<метод>_s<seed>.json`.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

KEYS = {"user": ["uid"], "item": ["item_id"], "ui": ["uid", "item_id"],
        "uart": ["uid", "artist_id"], "ualb": ["uid", "album_id"],
        "art": ["artist_id"], "alb": ["album_id"]}
AGGS = ["cnt", "rate", "mean_pr", "skip_rate", "full_rate", "recency"]
TARGET_OF = {"rate": "label", "mean_pr": "pr", "skip_rate": "skip", "full_rate": "full"}
NEW = {}            # имя признака -> значения для всех строк df
KEEP = set()        # признаки, принятые хотя бы одним запуском (их массивы нельзя удалять)

def validate(p):
    try:
        key, agg = str(p["key"]), str(p["agg"])
        w = int(p.get("window", 1)) if agg != "recency" else 0
    except Exception:
        return None
    if key not in KEYS or agg not in AGGS:
        return None
    return {"key": key, "agg": agg, "window": min(max(w, 1), 400) if agg != "recency" else 0}

def fname(p):
    return f"{p['key']}_recency_d" if p["agg"] == "recency" else f"{p['key']}_{p['agg']}_{p['window']}d"

_name_re = re.compile(r"^(user|item|ui|uart|ualb|art|alb)_(cnt|rate|mean_pr|skip_rate|full_rate)_(\d+)d$")
_rec_re = re.compile(r"^(user|item|ui|uart|ualb|art|alb)_recency_d$")
def parse_name(n):
    m = _name_re.match(n)
    if m:
        return validate({"key": m.group(1), "agg": m.group(2), "window": int(m.group(3))})
    m = _rec_re.match(n)
    if m:
        return validate({"key": m.group(1), "agg": "recency", "window": 0})
    return None

def compile_programs(progs):
    groups = {}
    for p in progs:
        if fname(p) in NEW:
            continue
        kind = "recency" if p["agg"] == "recency" else ("cnt" if p["agg"] == "cnt" else TARGET_OF[p["agg"]])
        groups.setdefault((p["key"], kind), []).append(p)
    for (key, kind), ps in groups.items():
        keys = KEYS[key]
        if kind == "recency":
            F = add_window_feats2(df, keys, [], key, recency=True)
            NEW[f"{key}_recency_d"] = F[f"{key}_recency_d"].to_numpy()
        elif kind == "cnt":
            ws = sorted({p["window"] for p in ps})
            F = add_window_feats2(df, keys, ws, key, target="label", with_cnt=True)
            for w in ws:
                NEW[f"{key}_cnt_{w}d"] = F[f"{key}_cnt_{w}d"].to_numpy()
        else:
            ws = sorted({p["window"] for p in ps})
            tag = "rate" if kind == "label" else f"mean_{kind}"
            F = add_window_feats2(df, keys, ws, key, target=kind, with_cnt=False)
            for p in ps:
                NEW[fname(p)] = F[f"{key}_{tag}_{p['window']}d"].to_numpy()

def ensure_features(names):
    progs = [parse_name(n) for n in names if n not in NEW]
    compile_programs([p for p in progs if p is not None])
    KEEP.update(names)

# --- быстрая модель-прокси ---
PROXY_FRAC, PROXY_TREES = 0.20, 60
ru = np.random.default_rng(7)
tr_users = d_tr["uid"].unique()
u_sub = set(ru.choice(tr_users, size=int(len(tr_users) * PROXY_FRAC), replace=False).tolist())
m_sub = d_tr["uid"].isin(u_sub).to_numpy()
y_sub = d_tr["label"].to_numpy()[m_sub]
sz_sub = d_tr.loc[m_sub].groupby("g", sort=True).size().to_numpy()
params_px = {**params_r, "learning_rate": 0.15, "num_leaves": 31, "seed": 0}
idx_tr_sub = idx_tr_all[m_sub]
ci_base = [CP[c] for c in FEATS]

def proxy_score(Xtr, Xsel, seed=0):
    dtr = lgb.Dataset(Xtr, y_sub, group=sz_sub, params={"max_bin": 31})
    m = lgb.train({**params_px, "seed": seed}, dtr, PROXY_TREES)
    return ndcg_at_k(d_sel["g"], m.predict(Xsel), d_sel["label"])

Xb_tr0, Xb_sel0 = Xtr_p[m_sub][:, ci_base], Xsel_p[:, ci_base]
noise = [proxy_score(Xb_tr0, Xb_sel0, seed=s) for s in (0, 1, 2)]
THR = max(0.0005, 2 * float(np.std(noise)))
print(f"прокси на M1: {np.mean(noise):.4f} ± {np.std(noise):.4f}; порог принятия Δ > {THR:.4f}")

# --- поставщики предложений ---
def sample_program(rng):
    return validate({"key": str(rng.choice(list(KEYS))), "agg": str(rng.choice(AGGS)),
                     "window": int(np.exp(rng.uniform(0, np.log(400))))})

def propose_random(history, n, seen, rng):
    out = []
    while len(out) < n:
        p = sample_program(rng)
        if fname(p) not in seen and all(fname(p) != fname(q) for q in out):
            out.append(p)
    return out

def propose_evo(history, n, seen, rng):
    good = [h["p"] for h in history if h["delta"] > 0]
    out, tries = [], 0
    while len(out) < n and tries < 200:
        tries += 1
        if good and rng.random() < 0.6:
            p = dict(good[int(rng.integers(len(good)))])
            r = rng.random()
            if r < 0.5 and p["agg"] != "recency":
                p["window"] = int(round(p["window"] * float(rng.choice([0.5, 0.7, 1.4, 2.0]))))
            elif r < 0.75:
                p["key"] = str(rng.choice(list(KEYS)))
            else:
                p["agg"] = str(rng.choice(AGGS))
            p = validate(p)
        else:
            p = sample_program(rng)
        if p and fname(p) not in seen and all(fname(p) != fname(q) for q in out):
            out.append(p)
    return out

def encode(p):
    v = [0.0] * (len(KEYS) + len(AGGS) + 1)
    v[list(KEYS).index(p["key"])] = 1.0
    v[len(KEYS) + AGGS.index(p["agg"])] = 1.0
    v[-1] = float(np.log1p(p["window"]))
    return v

def propose_surrogate(history, n, seen, rng, kappa=1.0, n_cand=300, warmup=16):
    if len(history) < warmup:
        return propose_random(history, n, seen, rng)
    X = np.array([encode(h["p"]) for h in history]); y = np.array([h["delta"] for h in history])
    rf = RandomForestRegressor(n_estimators=100, min_samples_leaf=2,
                               random_state=int(rng.integers(1_000_000))).fit(X, y)
    cands = propose_random(history, n_cand, seen, rng)
    Xc = np.array([encode(p) for p in cands])
    preds = np.stack([t.predict(Xc) for t in rf.estimators_])
    score = preds.mean(0) + kappa * preds.std(0)
    return [cands[i] for i in np.argsort(-score)[:n]]

def run_agent(propose, rounds=10, per_round=8, seed=0):
    rng = np.random.default_rng(seed)
    Xtr_b, Xsel_b = Xb_tr0.copy(), Xb_sel0.copy()
    base = float(np.mean(noise))
    seen, accepted, history, t0 = set(FEATS), [], [], time.time()
    for r in range(rounds):
        progs = propose(history, per_round, seen, rng)
        if not progs:
            print(f"  раунд {r+1}: нет предложений"); continue
        compile_programs(progs)
        best = None
        for p in progs:
            nm = fname(p); seen.add(nm)
            s = proxy_score(np.column_stack([Xtr_b, NEW[nm][idx_tr_sub]]),
                            np.column_stack([Xsel_b, NEW[nm][idx_sel]]))
            history.append({"p": p, "delta": float(s - base)})
            if best is None or s - base > best[1]:
                best = (p, s - base, s)
        if best[1] > THR:
            p, d_, s = best
            Xtr_b = np.column_stack([Xtr_b, NEW[fname(p)][idx_tr_sub]])
            Xsel_b = np.column_stack([Xsel_b, NEW[fname(p)][idx_sel]])
            accepted.append(fname(p)); KEEP.add(fname(p)); base = s
        for p in progs:
            if fname(p) not in KEEP:
                NEW.pop(fname(p), None)
        print(f"  раунд {r+1}: лучший {fname(best[0])} Δ={best[1]:+.4f}"
              f"{' (принят)' if best[1] > THR else ''}; принято {len(accepted)}", flush=True)
    return dict(accepted=accepted, history=history, evals=len(history), sec=time.time() - t0)

def run_agent_cached(name, prop, seed):
    path_ = f"results/agent_{name}_s{seed}.json"
    if os.path.exists(path_):
        res = json.load(open(path_)); ensure_features(res["accepted"]); return res
    res = run_agent(prop, ROUNDS, PER_ROUND, seed=seed)
    json.dump(res, open(path_, "w"), ensure_ascii=False)
    return res

def final_eval(names, seed=0):
    ensure_features(names)
    def X(Xp, idx): return np.column_stack([Xp[:, ci_base]] + [NEW[n][idx] for n in names])
    Xtr, Xs, Xr = X(Xtr_p, idx_tr_all), X(Xsel_p, idx_sel), X(Xrep_p, idx_rep)
    dtr = lgb.Dataset(Xtr, d_tr["label"].to_numpy(), group=sz_tr, params={"max_bin": 63})
    dsel = lgb.Dataset(Xs, d_sel["label"].to_numpy(), group=sz_sel, reference=dtr)
    m = lgb.train({**params_r, "seed": seed}, dtr, 300, valid_sets=[dsel],
                  callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(0)])
    p_rep = m.predict(Xr, num_iteration=m.best_iteration)
    return ndcg_at_k(d_rep["g"], p_rep, d_rep["label"]), p_rep

In [ ]:
multi = {}
if RUN_AGENTS:
    for name, prop in [("random", propose_random), ("evo", propose_evo), ("surrogate", propose_surrogate)]:
        multi[name] = []
        for s in SEEDS_A:
            print(name, "seed", s, flush=True)
            multi[name].append(run_agent_cached(name, prop, s))
    rows = []
    for name, runs in multi.items():
        for s, res in zip(SEEDS_A, runs):
            s_rep, p = final_eval(res["accepted"])
            mean, ci = boot_diff_rep(R["M1"]["p"], p)
            first = next((i + 1 for i, h in enumerate(res["history"]) if h["delta"] > THR), None)
            rows.append(dict(метод=name, seed=s, принято=len(res["accepted"]), оценок_до_первого=first,
                             rep=s_rep, d_M1=mean, сек=round(res["sec"])))
    summary = pd.DataFrame(rows)
    summary.to_csv("results/agents_summary.csv", index=False)
    print(summary.round(4).to_string(index=False))
    print(summary.groupby("метод")[["принято", "rep", "d_M1"]].agg(["mean", "std"]).round(4))
else:
    print("RUN_AGENTS = False: запуск агентов пропущен (кэшированные результаты подхватываются при RUN_AGENTS = True)")

## 5. Графовые признаки: вложения двудольного графа и «векторы вкуса»

Граф строится только по событиям **строго до начала окна обучения** (`T0 = t_val − 60 дней`), поэтому утечки нет. Результат кэшируется в `features_graph_v1.parquet`.

In [ ]:
import scipy.sparse as sp
from scipy.sparse.linalg import svds

def spectral_item_vectors(u, i, d=16, min_deg=5):
    ucodes, uidx = pd.factorize(u); icodes, iidx = pd.factorize(i)
    M = sp.coo_matrix((np.ones(len(ucodes), np.float32), (ucodes, icodes)),
                      shape=(len(uidx), len(iidx))).tocsr()
    M.data = np.minimum(M.data, 1.0)
    deg_i = np.asarray(M.sum(0)).ravel(); keep_ = deg_i >= min_deg
    M, iidx = M[:, keep_], iidx[keep_]
    du = np.asarray(M.sum(1)).ravel(); di = np.asarray(M.sum(0)).ravel()
    M = sp.diags(1 / np.sqrt(np.maximum(du, 1))) @ M @ sp.diags(1 / np.sqrt(np.maximum(di, 1)))
    _, S, Vt = svds(M.astype(np.float64), k=d)
    V = Vt.T * S
    V = V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-9)
    return pd.Series(np.arange(len(iidx)), index=pd.Index(iidx)), V.astype(np.float32)

def taste_feats(df, vec_index, V, windows_days, name, snap=DAY, chunk=1_000_000):
    n, d = len(df), V.shape[1]
    pos = vec_index.reindex(df["item_id"].to_numpy()).to_numpy(dtype=np.float64)
    has = ~np.isnan(pos)
    E = V[np.where(has, pos, 0).astype(np.int64)].astype(np.float32)
    E[~has] = 0.0
    codes = df.groupby("uid", sort=False).ngroup().to_numpy(np.int64)
    ts = df["timestamp"].to_numpy(np.int64)
    tq = (ts // snap) * snap
    comb = (codes << SHIFT) | ts
    order = np.argsort(comb, kind="stable")
    comb_s, codes_s, tq_s = comb[order], codes[order], tq[order]
    E_s, has_s = E[order], has[order]
    lab_s = df["label"].to_numpy()[order].astype(np.float64)
    hi = np.searchsorted(comb_s, (codes_s << SHIFT) | tq_s, side="left")
    los = {w: np.searchsorted(comb_s, (codes_s << SHIFT) | np.maximum(tq_s - w * DAY, 0), side="left")
           for w in windows_days}
    out = {}
    for sign, wt in (("pos", lab_s), ("neg", 1.0 - lab_s)):
        wt = wt * has_s
        cumw = np.concatenate([[0.0], np.cumsum(wt)])
        cumv = np.zeros((n + 1, d), np.float64)
        np.cumsum(E_s * wt[:, None], axis=0, out=cumv[1:])
        for w in windows_days:
            res_s = np.full(n, np.nan, np.float32)
            lo = los[w]
            for a in range(0, n, chunk):
                b = min(n, a + chunk)
                S = cumv[hi[a:b]] - cumv[lo[a:b]]
                W = cumw[hi[a:b]] - cumw[lo[a:b]]
                nrm = np.linalg.norm(S, axis=1)
                dot = np.einsum("ij,ij->i", S, E_s[a:b].astype(np.float64))
                ok = (W > 0) & (nrm > 1e-9) & has_s[a:b]
                res_s[a:b] = np.where(ok, dot / np.maximum(nrm, 1e-9), np.nan)
            r = np.empty(n, np.float32); r[order] = res_s
            out[f"{name}_taste_{sign}_{w}d"] = r
        del cumv
    res = pd.DataFrame(out, index=df.index)
    for w in windows_days:
        res[f"{name}_taste_diff_{w}d"] = (res[f"{name}_taste_pos_{w}d"] - res[f"{name}_taste_neg_{w}d"]).astype(np.float32)
    return res

GRAPH_PATH = "features_graph_v1.parquet"
cached = find_file(GRAPH_PATH, required=False)
if cached:
    TASTE = pd.read_parquet(cached)
    print("граф-признаки загружены из кэша:", cached)
else:
    RAW = find_file("multi_event.parquet", required=False)
    if RAW is None:
        from huggingface_hub import hf_hub_download
        RAW = hf_hub_download(repo_id="yandex/yambda", repo_type="dataset",
                              filename="flat/50m/multi_event.parquet", local_dir="data")
    T0 = int(t_val - TRAIN_DAYS * DAY)
    tab = pq.read_table(RAW, columns=["uid", "item_id", "timestamp", "played_ratio_pct"],
                        filters=[("event_type", "==", "listen"), ("timestamp", "<", T0),
                                 ("played_ratio_pct", ">=", POS_THRESHOLD)])
    past = tab.to_pandas(); del tab
    print(len(past), "дослушанных прослушиваний до T0")
    vec_index, V_graph = spectral_item_vectors(past["uid"].to_numpy(), past["item_id"].to_numpy(), d=16, min_deg=5)
    del past
    print("вложений треков:", V_graph.shape, "| покрытие строк df:", round(df["item_id"].isin(vec_index.index).mean(), 3))
    TASTE = taste_feats(df, vec_index, V_graph, [30, 90, 400], "graph")
    TASTE.to_parquet(GRAPH_PATH)
assert len(TASTE) == len(df)
print(TASTE.notna().mean().round(3))
for c in ["graph_taste_pos_90d", "graph_taste_neg_90d", "graph_taste_diff_90d"]:
    v = TASTE[c].to_numpy()[idx_rep]
    print(c, round(ndcg_at_k(d_rep["g"], np.nan_to_num(v, nan=np.nanmean(v)), d_rep["label"]), 4))

## 6. Проверка: добавляют ли граф-признаки качество на фоне насыщения агрегатов

Сравнение RFE на пуле v2 (база) и на пуле v2 + граф-признаки при K = 48 и 26 с доверительными интервалами по пользователям (rep).

In [ ]:
def register_features(names):
    global Xtr_p, Xsel_p, Xrep_p
    for nm in names:
        if nm in CP:
            continue
        CP[nm] = Xtr_p.shape[1]
        Xtr_p  = np.column_stack([Xtr_p,  NEW[nm][idx_tr_all]])
        Xsel_p = np.column_stack([Xsel_p, NEW[nm][idx_sel]])
        Xrep_p = np.column_stack([Xrep_p, NEW[nm][idx_rep]])

def compare_rfe(pool_cols, tag, base_sets, Ks=(48, 26)):
    t = time.time()
    new_sets = rfe_path(pool_cols, list(Ks))
    print(f"RFE на пуле '{tag}':", round(time.time() - t), "с")
    rows = []
    for K in Ks:
        for name, cols in [("база", base_sets[K]), (tag, new_sets[K])]:
            _, s_sel, s_rep, p = fit_sel(cols)
            R[f"{name}_{K}"] = dict(p=p, rep=s_rep)
            print(K, name, round(s_sel, 4), round(s_rep, 4))
        mean, ci = boot_diff_rep(R[f"база_{K}"]["p"], R[f"{tag}_{K}"]["p"])
        print(f"K={K}: {tag} - база = {mean:+.4f} [{ci[0]:+.4f}; {ci[1]:+.4f}]")
        rows.append(dict(K=K, d=mean, lo=ci[0], hi=ci[1], база=R[f"база_{K}"]["rep"], новый=R[f"{tag}_{K}"]["rep"]))
    pd.DataFrame(rows).to_csv(f"results/compare_rfe_{tag}.csv", index=False)
    return new_sets

for nm in TASTE.columns:
    NEW[nm] = TASTE[nm].to_numpy(); KEEP.add(nm)
taste_names = list(TASTE.columns)
register_features(taste_names)
POOL4_COLS = list(POOL2_COLS) + taste_names
sets4 = compare_rfe(POOL4_COLS, "пул+граф", sets_rfe)
json.dump({str(k): v for k, v in sets4.items()}, open("results/selected_sets_pool_graph.json", "w"), ensure_ascii=False, indent=1)

In [ ]:
# ===== диагностика: помогает ли граф на НОВЫХ для пользователя парах =====
mask_new = Xrep_p[:, CP["ui_cnt_400d"]] == 0          # пользователь не слушал этот трек за всю историю
print("доля новых пар на rep:", round(float(mask_new.mean()), 3))
g_rep, y_rep = d_rep["g"].to_numpy(), d_rep["label"].to_numpy()

def boot_diff_masked(pa, pb, mask, n_boot=1000, seed=0):
    ga = ndcg_groups(g_rep[mask], pa[mask], y_rep[mask]); gb = ndcg_groups(g_rep[mask], pb[mask], y_rep[mask])
    diff = (gb - ga).dropna()
    pu = diff.groupby(diff.index // 1000).agg(["sum", "count"])
    s, c = pu["sum"].to_numpy(), pu["count"].to_numpy(); r = np.random.default_rng(seed); est = []
    for _ in range(n_boot):
        i = r.integers(0, len(s), len(s)); est.append(s[i].sum() / c[i].sum())
    return diff.mean(), np.percentile(est, [2.5, 97.5])

for K in (48, 26):
    pa, pb = R[f"база_{K}"]["p"], R[f"пул+граф_{K}"]["p"]
    for nm, mk in [("новые пары", mask_new), ("повторные пары", ~mask_new)]:
        mean, ci = boot_diff_masked(pa, pb, mk)
        print(f"K={K}, {nm}: пул+граф - база = {mean:+.4f} [{ci[0]:+.4f}; {ci[1]:+.4f}]")

In [ ]:
def add_lastN_feats(df, keys, Ns, name, target="label", snap=DAY):
    """Для каждой строки: число и среднее target по ПОСЛЕДНИМ N событиям той же группы,
    случившимся строго до начала суток (снимок). Память на ключ ограничена N событиями."""
    codes = df.groupby(keys, sort=False).ngroup().to_numpy(np.int64)
    ts = df["timestamp"].to_numpy(np.int64)
    val = df[target].to_numpy(np.float64)
    tq = (ts // snap) * snap
    comb = (codes << SHIFT) | ts
    order = np.argsort(comb, kind="stable")
    comb_s, codes_s, tq_s, val_s = comb[order], codes[order], tq[order], val[order]
    cum = np.concatenate([[0.0], np.cumsum(val_s)])
    hi = np.searchsorted(comb_s, (codes_s << SHIFT) | tq_s, side="left")
    gstart = np.searchsorted(comb_s, codes_s << SHIFT, side="left")
    tag = "rate" if target == "label" else f"mean_{target}"
    out = {}
    for N in Ns:
        lo = np.maximum(hi - N, gstart)
        cnt = (hi - lo).astype(np.float64)
        mean = np.where(cnt > 0, (cum[hi] - cum[lo]) / np.maximum(cnt, 1), np.nan)
        r = np.empty(len(df), np.float32); r[order] = cnt; out[f"{name}_last{N}_cnt"] = r
        r = np.empty(len(df), np.float32); r[order] = mean; out[f"{name}_last{N}_{tag}"] = r
    return out

# проверка против наивного перебора
_r = np.random.default_rng(2); _n = 3000
_t = pd.DataFrame({"uid": _r.integers(0, 8, _n), "item_id": _r.integers(0, 12, _n),
                   "timestamp": _r.integers(0, 40 * DAY // 5, _n) * 5, "label": _r.integers(0, 2, _n)}
                  ).sort_values("timestamp").reset_index(drop=True)
_F = add_lastN_feats(_t, ["uid", "item_id"], [1, 3, 10], "ui")
for i in _r.choice(_n, 250, replace=False):
    row = _t.iloc[i]; tq = (row.timestamp // DAY) * DAY
    prev = _t[(_t.uid == row.uid) & (_t.item_id == row.item_id) & (_t.timestamp < tq)]
    for N in (1, 3, 10):
        last = prev.tail(N)
        assert _F[f"ui_last{N}_cnt"][i] == len(last)
        e = last.label.mean() if len(last) else np.nan; g = _F[f"ui_last{N}_rate"][i]
        assert (np.isnan(e) and np.isnan(g)) or abs(e - g) < 1e-6
print("тест add_lastN_feats пройден")

def register_batch(feat_dict):
    """Добавляет признаки (массивы для всех строк df) в матрицы train/sel/rep."""
    global Xtr_p, Xsel_p, Xrep_p
    names = [n for n in feat_dict if n not in CP]
    for n in names:
        CP[n] = Xtr_p.shape[1] + names.index(n)
    Xtr_p  = np.column_stack([Xtr_p]  + [feat_dict[n][idx_tr_all] for n in names])
    Xsel_p = np.column_stack([Xsel_p] + [feat_dict[n][idx_sel] for n in names])
    Xrep_p = np.column_stack([Xrep_p] + [feat_dict[n][idx_rep] for n in names])
    return names

In [ ]:
_n0, _cp0 = Xtr_p.shape[1], dict(CP)               # запомним состояние матриц, чтобы потом вернуть
BUDGETS = [1, 3, 10, 30, 100, 300, 1000]
HIST_KEYS = {"user": ["uid"], "ui": ["uid", "item_id"], "uart": ["uid", "artist_id"], "ualb": ["uid", "album_id"]}
hist_names = []
for kname, kcols in HIST_KEYS.items():
    for target in ("label", "pr"):
        d_ = add_lastN_feats(df, kcols, BUDGETS, kname, target=target)
        if target == "pr":                                 # cnt уже посчитан для label
            d_ = {k: v for k, v in d_.items() if not k.endswith("_cnt")}
        hist_names += register_batch(d_)
        del d_; gc.collect()
print(len(hist_names), "признаков истории по последним N событиям")

base_item = [c for c in POOL2_COLS if c.startswith(("item_", "gitem_"))]     # популярность трека
def hist_cols(B):
    return [n for n in hist_names if int(re.search(r"_last(\d+)_", n).group(1)) <= B]

rows = []
for B in BUDGETS:
    cols = base_item + hist_cols(B)
    _, s_sel, s_rep, _ = fit_sel(cols)
    rows.append(dict(B=B, признаков=len(cols), sel=s_sel, rep=s_rep))
    print(B, len(cols), round(s_sel, 4), round(s_rep, 4), flush=True)
budget_df = pd.DataFrame(rows); budget_df.to_csv("results/history_budget.csv", index=False)

import matplotlib.pyplot as plt
plt.figure(figsize=(7, 4))
plt.semilogx(budget_df["B"], budget_df["rep"], "o-", label="история: последние B событий (+ популярность трека)")
for nm, ls in [("M1", ":"), ("M2", "--"), ("POOL2", "-.")]:
    plt.axhline(R[nm]["rep"], ls=ls, color="gray", label=nm)
plt.xlabel("бюджет истории B (число последних событий на ключ)"); plt.ylabel("NDCG@10 (rep)")
plt.grid(alpha=.3); plt.legend(fontsize=8)
plt.savefig("history_budget.png", dpi=150, bbox_inches="tight"); plt.show()

# возвращаем матрицы в исходное состояние (признаки истории нужны только для этого эксперимента)
Xtr_p  = np.ascontiguousarray(Xtr_p[:, :_n0]); Xsel_p = np.ascontiguousarray(Xsel_p[:, :_n0]); Xrep_p = np.ascontiguousarray(Xrep_p[:, :_n0])
CP.clear(); CP.update(_cp0); gc.collect()
print("матрицы восстановлены:", Xtr_p.shape[1], "столбцов")

In [ ]:
def time_generator(ns, reps=2):
    out = []
    for n in ns:
        sub = df.iloc[:n].reset_index(drop=True)
        ts_ = []
        for _ in range(reps):
            t = time.time()
            add_window_feats2(sub, ["uid", "item_id"], [7, 30, 90], "ui", with_cnt=True)
            ts_.append(time.time() - t)
        out.append((n, min(ts_)))
        print(n, f"{min(ts_):.2f} с", flush=True)
    return pd.DataFrame(out, columns=["n", "сек"])
ns = [int(len(df) * f) for f in (0.0625, 0.125, 0.25, 0.5, 1.0)]
scal = time_generator(ns)
slope = np.polyfit(np.log(scal["n"]), np.log(scal["сек"]), 1)[0]
print(f"наклон log(время) от log(n): {slope:.2f} (1,0 = линейно; чуть больше 1 = n log n)")
scal.to_csv("results/scalability.csv", index=False)

In [ ]:
import glob
def register_features(names):
    global Xtr_p, Xsel_p, Xrep_p
    for nm in names:
        if nm in CP:
            continue
        CP[nm] = Xtr_p.shape[1]
        Xtr_p  = np.column_stack([Xtr_p,  NEW[nm][idx_tr_all]])
        Xsel_p = np.column_stack([Xsel_p, NEW[nm][idx_sel]])
        Xrep_p = np.column_stack([Xrep_p, NEW[nm][idx_rep]])

names_acc = sorted({n for f in glob.glob("results/agent_*_s*.json") for n in json.load(open(f))["accepted"]})
ensure_features(names_acc); register_features(names_acc)
print(len(names_acc), "признаков агентов зарегистрировано")

In [ ]:
def get_pred(key, cols):
    if key not in R or "p" not in R[key]:
        _, _, s_rep, p = fit_sel(list(cols)); R[key] = dict(p=p, rep=s_rep)
    return R[key]["p"]
P_M1, P_M2   = get_pred("M1", FEATS), get_pred("M2", M2_COLS)
P_POOL       = get_pred("POOL2", POOL2_COLS)
P_R26        = get_pred("RFE26_v2", sets_rfe[26])

g_rep, y_rep = d_rep["g"].to_numpy(), d_rep["label"].to_numpy()
def strat_diff(pa, pb, mask, n_boot=500, seed=0, min_groups=30):
    ga = ndcg_groups(g_rep[mask], pa[mask], y_rep[mask]); gb = ndcg_groups(g_rep[mask], pb[mask], y_rep[mask])
    diff = (gb - ga).dropna()
    if len(diff) < min_groups:
        return None
    pu = diff.groupby(diff.index // 1000).agg(["sum", "count"])
    s, c = pu["sum"].to_numpy(), pu["count"].to_numpy(); r = np.random.default_rng(seed); est = []
    for _ in range(n_boot):
        i = r.integers(0, len(s), len(s)); est.append(s[i].sum() / c[i].sum())
    lo, hi = np.percentile(est, [2.5, 97.5])
    return dict(групп=len(diff), A=ga.mean(), B=gb.mean(), d=diff.mean(), lo=lo, hi=hi)

col = lambda name: Xrep_p[:, CP[name]]
def terciles(x):
    q1, q2 = np.nanquantile(x, [1/3, 2/3])
    return [("низкая", x <= q1), ("средняя", (x > q1) & (x <= q2)), ("высокая", x > q2)]
strata = [("пары: новые (ui_cnt_400d=0)", col("ui_cnt_400d") == 0), ("пары: повторные", col("ui_cnt_400d") > 0)]
strata += [(f"популярность трека (gitem_cnt_90d): {n}", m) for n, m in terciles(col("gitem_cnt_90d"))]
strata += [(f"активность пользователя (user_cnt_30d): {n}", m) for n, m in terciles(col("user_cnt_30d"))]

rows = []
for cmp_name, pa, pb in [("RFE-26 − M1", P_M1, P_R26), ("пул v2 − M2", P_M2, P_POOL)]:
    for sname, mk in strata:
        r = strat_diff(pa, pb, mk)
        if r: rows.append(dict(сравнение=cmp_name, страта=sname, доля_строк=round(float(mk.mean()), 3), **r))
strat_df = pd.DataFrame(rows)
print(strat_df.round(4).to_string(index=False))
strat_df.to_csv("results/strata_diff.csv", index=False)

# устойчивость по времени: недели валидации (rep)
wk = d_rep["timestamp"].to_numpy().astype(np.int64) // (7 * DAY)
rows = []
for w in sorted(set(wk)):
    mk = wk == w
    for cmp_name, pa, pb in [("RFE-26 − M1", P_M1, P_R26), ("пул v2 − M2", P_M2, P_POOL)]:
        r = strat_diff(pa, pb, mk)
        if r: rows.append(dict(неделя=int(w), сравнение=cmp_name, доля_строк=round(float(mk.mean()), 3), **r))
week_df = pd.DataFrame(rows)
print(week_df.round(4).to_string(index=False))
week_df.to_csv("results/weekly_diff.csv", index=False)

In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
rs_ = np.random.default_rng(0)
ns_ = min(300_000, Xtr_p.shape[0]); rows_ = rs_.choice(Xtr_p.shape[0], ns_, replace=False)
names_all = sorted(CP, key=CP.get)
sub = pd.DataFrame(Xtr_p[rows_][:, [CP[n] for n in names_all]], columns=names_all)
corr = sub.rank(method="average", na_option="keep").corr(min_periods=1000).abs().fillna(0.0)   # ранги -> корреляция Спирмена

pool_names = list(POOL2_COLS); new_names = [n for n in names_all if n not in pool_names]
Dm = 1 - corr.loc[pool_names, pool_names].to_numpy(); np.fill_diagonal(Dm, 0)
Z = linkage(squareform((Dm + Dm.T) / 2, checks=False), method="average")
for thr in (0.95, 0.9, 0.8):
    print(f"пул v2: кластеров при |ρ|>{thr}: {fcluster(Z, 1 - thr, criterion='distance').max()} из {len(pool_names)}")
if new_names:
    rows = []
    for n in new_names:
        c = corr.loc[n, pool_names]
        rows.append(dict(признак=n, max_abs_rho=float(c.max()), ближайший=c.idxmax()))
    red = pd.DataFrame(rows).sort_values("max_abs_rho", ascending=False)
    print(red.round(3).to_string(index=False))
    print("доля новых признаков с max|ρ|>0,8 относительно пула v2:", round(float((red["max_abs_rho"] > 0.8).mean()), 3))
    red.to_csv("results/redundancy_new_vs_pool.csv", index=False)
else:
    print("новых признаков (агенты, граф) в матрицах нет: зарегистрируйте их (register_features) и повторите")

In [ ]:
N_RES = 5                         # число подвыборок пользователей (по 80%)
def fit_gain_sub(cols, rows_mask, seed=0):
    ci = [CP[c] for c in cols]
    d_ = d_tr.loc[rows_mask]; sz_ = d_.groupby("g", sort=True).size().to_numpy()
    dtr  = lgb.Dataset(Xtr_p[rows_mask][:, ci], d_["label"].to_numpy(), group=sz_, params={"max_bin": 63})
    dsel = lgb.Dataset(Xsel_p[:, ci], d_sel["label"].to_numpy(), group=sz_sel, reference=dtr)
    m = lgb.train({**params_r, "seed": seed}, dtr, 300, valid_sets=[dsel],
                  callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(0)])
    return pd.Series(m.feature_importance("gain"), index=cols)

def rfe_path_sub(start_cols, targets, rows_mask, drop_frac=0.15):
    cols = list(start_cols); out = {}
    for K in sorted(targets, reverse=True):
        while len(cols) > K:
            imp = fit_gain_sub(cols, rows_mask).sort_values(ascending=False)
            cols = imp.index[:max(K, int(len(cols) * (1 - drop_frac)))].tolist()
        out[K] = list(cols)
    return out

users_tr = d_tr["uid"].unique()
res_sets = {26: [], 48: []}
for r in range(N_RES):
    ur = np.random.default_rng(100 + r)
    keep_u = set(ur.choice(users_tr, size=int(0.8 * len(users_tr)), replace=False).tolist())
    mk = d_tr["uid"].isin(keep_u).to_numpy()
    t = time.time(); sets_r = rfe_path_sub(POOL2_COLS, [48, 26], mk)
    for K in (48, 26): res_sets[K].append(set(sets_r[K]))
    print(f"подвыборка {r+1}/{N_RES}: {round(time.time()-t)} с", flush=True)

def jacc(a, b): return len(a & b) / len(a | b)
stab = []
for K in (26, 48):
    S = res_sets[K]; ref = set(sets_rfe[K])
    pair = [jacc(S[i], S[j]) for i in range(len(S)) for j in range(i + 1, len(S))]
    vs_ref = [jacc(s_, ref) for s_ in S]
    freq = pd.Series({f: sum(f in s_ for s_ in S) / len(S) for f in POOL2_COLS}).sort_values(ascending=False)
    core = int((freq == 1.0).sum())
    stab.append(dict(K=K, жаккар_между_подвыборками=np.mean(pair), жаккар_с_полным_RFE=np.mean(vs_ref), ядро_во_всех_подвыборках=core))
    freq.head(15).to_csv(f"results/selection_frequency_K{K}.csv")
stab_df = pd.DataFrame(stab); print(stab_df.round(3).to_string(index=False)); stab_df.to_csv("results/selection_stability.csv", index=False)

## Что дальше
- Если для «пул+граф минус база» интервал не содержит нуля и разность положительна: граф-признаки включаем в финальные наборы.
- Если интервал включает ноль: фиксируем как отрицательный результат (статичный граф, d=16, окна 30/90/400 дней).
- Затем: заморозка финалистов, шум по seed, один запуск на test.

In [ ]:
graph_names = [n for n in CP if n.startswith("graph_taste")]
rs = np.random.default_rng(0)
users_ = d_tr["uid"].unique()
hold = set(rs.choice(users_, size=int(0.2 * len(users_)), replace=False).tolist())
is_hold = d_tr["uid"].isin(hold).to_numpy()
pool_idx = [CP[c] for c in POOL2_COLS]
res = []
for gname in graph_names:
    y = Xtr_p[:, CP[gname]]; ok = ~np.isnan(y)
    tr_i = np.where(ok & ~is_hold)[0]; ho_i = np.where(ok & is_hold)[0]
    tr_i = rs.choice(tr_i, size=min(300_000, len(tr_i)), replace=False)
    ho_i = rs.choice(ho_i, size=min(100_000, len(ho_i)), replace=False)
    m = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.1, num_leaves=63, subsample=0.8, subsample_freq=1,
                          colsample_bytree=0.8, verbose=-1)
    m.fit(Xtr_p[tr_i][:, pool_idx], y[tr_i])
    pred = m.predict(Xtr_p[ho_i][:, pool_idx])
    r2 = 1 - np.sum((y[ho_i] - pred) ** 2) / np.sum((y[ho_i] - y[ho_i].mean()) ** 2)
    res.append(dict(признак=gname, R2_на_отложенных_пользователях=float(r2)))
    print(gname, round(float(r2), 3), flush=True)
pd.DataFrame(res).to_csv("results/conditional_redundancy_graph.csv", index=False)

In [ ]:
# ===== L1: явная обратная связь (лайки, дизлайки) для пользователей подвыборки =====
RAW = find_file("multi_event.parquet", required=False)
if RAW is None:
    from huggingface_hub import hf_hub_download
    RAW = hf_hub_download(repo_id="yandex/yambda", repo_type="dataset",
                          filename="flat/50m/multi_event.parquet", local_dir="data")
tab = pq.read_table(RAW, columns=["uid", "item_id", "timestamp", "event_type"],
                    filters=[("event_type", "in", ["like", "unlike", "dislike", "undislike"]),
                             ("uid", "in", df["uid"].unique().tolist())])
ev = tab.to_pandas(); del tab
print(ev["event_type"].value_counts())
amap, bmap = load_map("artist_item_mapping.parquet"), load_map("album_item_mapping.parquet")
for col, mp in [("artist_id", amap), ("album_id", bmap)]:
    s = ev["item_id"].map(mp).astype("float64")
    ev[col] = np.where(s.isna(), -1 - ev["item_id"].astype(np.int64), s).astype(np.int64)
ev["sign"] = ev["event_type"].map({"like": 1.0, "unlike": -1.0, "dislike": 1.0, "undislike": -1.0})
ev["stream"] = ev["event_type"].map({"like": "like", "unlike": "like", "dislike": "dislike", "undislike": "dislike"})

# ===== L2: признаки по типам событий и отношения =====
def feedback_feats(df, ev_s, keys, windows_days, name, snap=DAY):
    """Чистое число лайков (или дизлайков) за окно до начала суток: (+1 за like, -1 за unlike) по ключу keys."""
    n = len(df)
    allk = pd.concat([df[keys], ev_s[keys]], ignore_index=True)
    codes_all = allk.groupby(keys, sort=False).ngroup().to_numpy(np.int64)
    cq, ce = codes_all[:n], codes_all[n:]
    te = ev_s["timestamp"].to_numpy(np.int64); val = ev_s["sign"].to_numpy(np.float64)
    comb_e = (ce << SHIFT) | te
    order = np.argsort(comb_e, kind="stable")
    comb_s, val_s = comb_e[order], val[order]
    cum = np.concatenate([[0.0], np.cumsum(val_s)])
    tq = (df["timestamp"].to_numpy(np.int64) // snap) * snap
    hi = np.searchsorted(comb_s, (cq << SHIFT) | tq, side="left")
    out = {}
    for w in windows_days:
        lo = np.searchsorted(comb_s, (cq << SHIFT) | np.maximum(tq - w * DAY, 0), side="left")
        out[f"{name}_net_{w}d"] = (cum[hi] - cum[lo]).astype(np.float32)
    return out

# проверка против наивного перебора
_r = np.random.default_rng(3); _n = 2500
_t = pd.DataFrame({"uid": _r.integers(0, 8, _n), "item_id": _r.integers(0, 12, _n),
                   "timestamp": _r.integers(0, 40 * DAY // 5, _n) * 5}).sort_values("timestamp").reset_index(drop=True)
_e = pd.DataFrame({"uid": _r.integers(0, 8, 400), "item_id": _r.integers(0, 12, 400),
                   "timestamp": _r.integers(0, 40 * DAY // 5, 400) * 5, "sign": _r.choice([1.0, -1.0], 400, p=[.8, .2])})
_F = feedback_feats(_t, _e, ["uid", "item_id"], [7, 30], "ui_like")
for i in _r.choice(_n, 250, replace=False):
    row = _t.iloc[i]; tq = (row.timestamp // DAY) * DAY
    for w in (7, 30):
        m = (_e.uid == row.uid) & (_e.item_id == row.item_id) & (_e.timestamp < tq) & (_e.timestamp >= tq - w * DAY)
        assert abs(_F[f"ui_like_net_{w}d"][i] - _e.sign[m].sum()) < 1e-6
print("тест feedback_feats пройден")

FB = {}
WFB = [30, 90, 400]
KEYS_FB = {"ui": ["uid", "item_id"], "uart": ["uid", "artist_id"], "ualb": ["uid", "album_id"], "user": ["uid"]}
for kname, kcols in KEYS_FB.items():
    for stream in ("like", "dislike"):
        FB.update(feedback_feats(df, ev[ev["stream"] == stream], kcols, WFB, f"{kname}_{stream}"))
# отношения: лайки на одно прослушивание (в духе «конверсии» из условия задания)
for kname in ("user", "uart", "ualb"):
    lc = add_window_feats2(df, KEYS_FB[kname], WFB, kname, with_cnt=True)
    for w in WFB:
        FB[f"{kname}_like_per_listen_{w}d"] = (FB[f"{kname}_like_net_{w}d"] / (lc[f"{kname}_cnt_{w}d"].to_numpy() + 1.0)).astype(np.float32)
    del lc
print(len(FB), "признаков обратной связи:", list(FB)[:6], "...")
for c in ["ui_like_net_400d", "uart_like_net_400d", "user_like_per_listen_400d"]:
    v = FB[c][idx_rep]
    print(c, "ненулевых:", round(float((v != 0).mean()), 4), "| NDCG@10 одной фичи:",
          round(ndcg_at_k(d_rep["g"], v + 1e-9 * np.random.default_rng(0).random(len(v)), d_rep["label"]), 4))

# ===== L3: добавить в пул и сравнить RFE на пуле v2 и на пуле v2 + обратная связь =====
def register_dict(feat_dict):
    global Xtr_p, Xsel_p, Xrep_p
    names = [n for n in feat_dict if n not in CP]
    for k, n in enumerate(names):
        CP[n] = Xtr_p.shape[1] + k
    Xtr_p  = np.column_stack([Xtr_p]  + [feat_dict[n][idx_tr_all] for n in names])
    Xsel_p = np.column_stack([Xsel_p] + [feat_dict[n][idx_sel] for n in names])
    Xrep_p = np.column_stack([Xrep_p] + [feat_dict[n][idx_rep] for n in names])
    return names

def compare_rfe_fb(pool_cols, tag, base_sets, Ks=(48, 26)):
    new_sets = rfe_path(pool_cols, list(Ks)); rows = []
    for K in Ks:
        for nm, cols in [("база", base_sets[K]), (tag, new_sets[K])]:
            _, s_sel, s_rep, p = fit_sel(cols); R[f"{nm}_{K}_fb"] = dict(p=p, rep=s_rep)
            print(K, nm, round(s_sel, 4), round(s_rep, 4))
        mean, ci = boot_diff_rep(R[f"база_{K}_fb"]["p"], R[f"{tag}_{K}_fb"]["p"])
        print(f"K={K}: {tag} - база = {mean:+.4f} [{ci[0]:+.4f}; {ci[1]:+.4f}]")
        rows.append(dict(K=K, d=mean, lo=ci[0], hi=ci[1], база=R[f"база_{K}_fb"]["rep"], новый=R[f"{tag}_{K}_fb"]["rep"]))
    pd.DataFrame(rows).to_csv(f"results/compare_rfe_{tag}.csv", index=False)
    return new_sets

fb_names = register_dict(FB)
POOL5_COLS = list(POOL2_COLS) + fb_names
sets5 = compare_rfe_fb(POOL5_COLS, "пул+обратная_связь", sets_rfe)
print("новые признаки в отобранном наборе K=26:", [c for c in sets5[26] if c in fb_names])